# Experiments

## Setup: Import Libraries and Scripts

In [9]:
import pandas as pd
from IPython.display import display, HTML
import optimize_prompt as opt  # Full optimization script
import zero_shot_baseline as zsb  # Zero-shot baseline script
import pickle
import os
from datetime import datetime

# Style for better table display
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_colwidth', None)  # Show full content in cells

# Path for saving experiment runs
RUNS_PICKLE_PATH = 'experiment_runs.pkl'

## Define Experiments

Add/edit experiments here. Each is a dict with:
- `'name'`: A label for the experiment.
- `'script'`: 'optimize' or 'zero_shot'.
- Other keys: Parameters for main() (e.g., generations, model_name).

In [ ]:
experiments = [
    {
        'name': 'Full Optimization - w META INSTRUCTION 2 & META TEMPLATE 2',
        'script': 'optimize',
        'generations': 10,
        'pop_size': 8,
        'train_sample_size': 50,
        'test_sample_size': 1000,
        'model_name': 'google/gemini-2.5-flash-lite-preview-06-17',
        'use_bandit_instr': True,
        'use_bandit_template': True,
        'statutory_context_enabled': True,
        'contract_context_enabled': True
    },
    {
        'name': 'Full Optimization - w META INSTRUCTION 2 & META TEMPLATE 2 - No Bandit',
        'script': 'optimize',
        'generations': 10,
        'pop_size': 8,
        'train_sample_size': 50,
        'test_sample_size': 1000,
        'model_name': 'google/gemini-2.5-flash-lite-preview-06-17',
        'use_bandit_instr': False,
        'use_bandit_template': False,
        'statutory_context_enabled': True,
        'contract_context_enabled': True
    },
]

## Run Experiments

This cell runs each experiment and collects results, saving them to a pickle file with a timecode.

In [11]:
results = []

for exp in experiments:
    print(f"\n=== Running Experiment: {exp['name']} ===")
    script = exp.pop('script')  # Remove script key for passing to main
    name = exp.pop('name')  # Remove name for passing to main
    run_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    
    try:
        if script == 'optimize':
            result = opt.main(**exp)
        elif script == 'zero_shot':
            result = zsb.main(**exp)
        else:
            raise ValueError(f"Unknown script: {script}")
        
        # Flatten metrics for table
        metrics = result['test_metrics']
        flat_result = {
            'Experiment Name': name,
            'Script': script,
            **exp,  # Add back parameters
            'Best Instruction': result['best_instruction'],
            'Best Template': result['best_template'],
            'Sample Size': metrics['sample_size'],
            'Valid Predictions': metrics['valid_predictions'],
            'Total Predictions': metrics['total_predictions'],
            'Accuracy': metrics['accuracy'],
            'Precision': metrics['precision'],
            'Recall': metrics['recall'],
            'F1 Micro': metrics['f1_micro'],
            'F1 Macro': metrics['f1_macro'],
            'Support (0/1)': f"{metrics['support'].get('0', 0)} / {metrics['support'].get('1', 0)}",
            'Unique y_true': ', '.join(metrics['unique_y_true']),
            'Unique y_pred': ', '.join(metrics['unique_y_pred']),
            'Detailed Report': metrics['detailed_report_string'],  # Full string for details
            'Full Classification Report (Dict)': metrics['classification_report'],  # Raw dict if needed
            'Run Time': run_time
        }
        results.append(flat_result)
    except Exception as e:
        print(f"Error in experiment '{name}': {e}")
        results.append({'Experiment Name': name, 'Error': str(e), 'Run Time': run_time})

# Load previous runs if exists
if os.path.exists(RUNS_PICKLE_PATH):
    with open(RUNS_PICKLE_PATH, 'rb') as f:
        past_runs = pickle.load(f)
else:
    past_runs = []

# Add new results to past runs and save
all_runs = past_runs + results
with open(RUNS_PICKLE_PATH, 'wb') as f:
    pickle.dump(all_runs, f)


=== Running Experiment: Full Optimization - w META INSTRUCTION 2 & META TEMPLATE 2 ===
Generation 1


KeyboardInterrupt: 

## Display Results Table

Interactive table with all parameters and metrics. Sorted by most recent run.

In [ ]:
# Load all runs from pickle and display sorted by most recent run time
import pickle
import pandas as pd
from IPython.display import display, HTML

RUNS_PICKLE_PATH = 'experiment_runs.pkl'

if os.path.exists(RUNS_PICKLE_PATH):
    with open(RUNS_PICKLE_PATH, 'rb') as f:
        all_runs = pickle.load(f)
    # Sort by 'Run Time' descending
    all_runs_sorted = sorted(all_runs, key=lambda x: x.get('Run Time', ''), reverse=True)
    df_results = pd.DataFrame(all_runs_sorted)
    if not df_results.empty:
        styled_df = df_results.style.set_properties(**{'text-align': 'left', 'white-space': 'pre-wrap'}).set_table_styles([
            {'selector': 'th', 'props': [('text-align', 'left')]}
        ]).background_gradient(cmap='viridis', subset=['F1 Macro'])
        display(HTML("<h3>All Experiment Runs (Most Recent First)</h3>"))
        display(styled_df)
    else:
        print("No results to display.")
else:
    print("No experiment runs found.")

,Experiment Name,Script,test_sample_size,model_name,statutory_context_enabled,contract_context_enabled,Best Instruction,Best Template,Sample Size,Valid Predictions,Total Predictions,Accuracy,Precision,Recall,F1 Micro,F1 Macro,Support (0/1),Unique y_true,Unique y_pred,Detailed Report,Full Classification Report (Dict),Run Time
0,zero-shot baseline,zero_shot,100,google/gemini-2.5-flash-lite-preview-06-17,True,True,Classify the following clause from a Terms of Service contract as fair (0) or unfair (1) using the statutory context and contract context for better understanding. Respond only with '0' or '1'.,Instruction: Clause: Statutory Context: Contract Context:,100,100,100,0.750000,0.178571,0.714286,0.750000,0.567100,93.0 / 7.0,"1, 0","1, 0",precision recall f1-score support 0 0.9722 0.7527 0.8485 93 1 0.1786 0.7143 0.2857 7 accuracy 0.7500 100 macro avg 0.5754 0.7335 0.5671 100 weighted avg 0.9167 0.7500 0.8091 100,"{'0': {'precision': 0.9722222222222222, 'recall': 0.7526881720430108, 'f1-score': 0.8484848484848485, 'support': 93.0}, '1': {'precision': 0.17857142857142858, 'recall': 0.7142857142857143, 'f1-score': 0.2857142857142857, 'support': 7.0}, 'accuracy': 0.75, 'macro avg': {'precision': 0.5753968253968254, 'recall': 0.7334869431643625, 'f1-score': 0.5670995670995671, 'support': 100.0}, 'weighted avg': {'precision': 0.9166666666666667, 'recall': 0.75, 'f1-score': 0.8090909090909091, 'support': 100.0}}",2025-07-22 13:17:31
